In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision.datasets import MNIST
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


In [2]:
device = torch.device("cpu" if torch.cuda.is_available() else "cpu")

# Transforms (better normalization for MNIST)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Dataset
trainset = MNIST(root="./MNIST_data", train=True, download=True, transform=transform)
testset = MNIST(root="./MNIST_data", train=False, download=True, transform=transform)

# DataLoader
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64, shuffle=False)

# Build our CNN

In [3]:
# CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  # (1,28,28) -> (32,28,28)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),              # -> (32,14,14)

            nn.Conv2d(32, 64, 3, padding=1), # -> (64,14,14)
            nn.ReLU(),
            nn.MaxPool2d(2, 2)               # -> (64,7,7)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


In [4]:
model = CNN().to(device)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Training
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_training_loss/len(train_loader):.4f}")

Epoch [1/10], Loss: 0.1302
Epoch [2/10], Loss: 0.0401
Epoch [3/10], Loss: 0.0285
Epoch [4/10], Loss: 0.0203
Epoch [5/10], Loss: 0.0149
Epoch [6/10], Loss: 0.0116
Epoch [7/10], Loss: 0.0116
Epoch [8/10], Loss: 0.0079
Epoch [9/10], Loss: 0.0085
Epoch [10/10], Loss: 0.0051


In [7]:
# Testing / Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")


Test Accuracy: 99.23%


In [8]:
torch.save(model.state_dict(), "mnist_cnn.pth")